# The Transformer

Tutorial of Computational Linguistics, National Chengchi University

*Chang-Yu Tsai, 2025.04.18*

- In this week, we will try:
  - to build a model based on the encoder-decoder strucutre
  - to build a model based on the transformer structure
  


## Set-up

- importing required packages

```
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence

import json

```


In [ ]:
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence

import json

## Preprocessing

- download the dataset from `Github`

```
!wget https://raw.githubusercontent.com/EntropiaTsai/nccu_elt_course_material/refs/heads/main/sentence_pairs_6000_mixed.json
```

In [ ]:
!wget https://raw.githubusercontent.com/EntropiaTsai/nccu_elt_course_material/refs/heads/main/sentence_pairs_6000_mixed.json

--2025-04-18 03:26:16--  https://raw.githubusercontent.com/EntropiaTsai/nccu_elt_course_material/refs/heads/main/sentence_pairs_6000_mixed.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4278 (4.2K) [text/plain]
Saving to: ‘sentence_pairs_6000_mixed.json.2’

sentence_pairs_6000 100%[===================>]   4.18K  --.-KB/s    in 0s      

2025-04-18 03:26:16 (63.7 MB/s) - ‘sentence_pairs_6000_mixed.json.2’ saved [4278/4278]



- reading the file

```
with open("sentence_pairs_6000_mixed.json", "r", encoding="utf-8") as f:
    data = json.load(f)
data
```

In [ ]:
with open("sentence_pairs_6000_mixed.json", "r", encoding="utf-8") as f:
    data = json.load(f)
data

{'他說可以': '他說可以',
 '他說不重要': '他說真的希望',
 '他說隨便': '他說真的希望',
 '大致上不重要': '大致上其實討厭',
 '大致上沒關係': '大致上真的希望',
 '理論上不重要': '理論上真的希望',
 '可能可以': '可能可以',
 '其實沒關係': '其實其實是',
 '理論上我沒事': '理論上我沒事',
 '大致上還好': '大致上不想',
 '我覺得可以': '我覺得不想',
 '我覺得隨便': '我覺得其實是',
 '我覺得不重要': '我覺得其實討厭',
 '可能沒關係': '可能其實是',
 '大致上隨便': '大致上隨便',
 '可能不重要': '可能可能會',
 '他說也行': '他說不太想',
 '感覺上也行': '感覺上其實是',
 '其實可以': '其實其實很難過',
 '看起來再說': '看起來不想',
 '看起來也行': '看起來也行',
 '理論上也行': '理論上其實是',
 '看起來都可以': '看起來不會',
 '看起來還好': '看起來不會',
 '表面上還好': '表面上其實很難過',
 '他說都可以': '他說不會',
 '原則上隨便': '原則上不想',
 '理論上可以': '理論上可以',
 '我覺得都可以': '我覺得都可以',
 '大致上可以': '大致上不想',
 '大致上都可以': '大致上都可以',
 '原則上可以': '原則上不想',
 '原則上我沒事': '原則上不會',
 '感覺上不重要': '感覺上可能會',
 '其實隨便': '其實不會',
 '他說再說': '他說不想',
 '理論上都可以': '理論上不太想',
 '我覺得也行': '我覺得真的希望',
 '其實不可以': '其實很重要',
 '可能再說': '可能其實是',
 '大致上我沒事': '大致上其實討厭',
 '原則上不可以': '原則上很重要',
 '看起來可以': '看起來很在意',
 '他說不可以': '他說不可以',
 '其實不重要': '其實不重要',
 '大致上也行': '大致上其實討厭',
 '看起來不可以': '看起來其實是',
 '感覺上都可以': '感覺上真的希望',
 '看起來我沒事': '看起來我沒事',
 '其實也行': '其實不想',
 '可能都可以': '可能很

### Text Encoding

- creating a list to map each character to an ID

  - It is common to use two special tokens in a decoder:
    
    - `BOS`: the beginning of the sentence
    - `EOS`: the ending of the sentence

```
special_tokens = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
all_chars=set()
for key, value in data.items():
  for combined in key + value:
    all_chars.add(combined)

char_list = special_tokens + sorted(all_chars)

char2ID = {}
for idx, char in enumerate(char_list):
  char2ID[char] = idx
ID2char = {}
for ch, i in char2ID.items():
  ID2char[i] = ch
```


In [ ]:
special_tokens = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
all_chars=set()
for key, value in data.items():
  for combined in key + value:
    all_chars.add(combined)

char_list = special_tokens + sorted(all_chars)

char2ID = {}
for idx, char in enumerate(char_list):
  char2ID[char] = idx
ID2char = {}
for ch, i in char2ID.items():
  ID2char[i] = ch

- converting each sentence into a string of IDs

  - for each sentence pair `data`:
    - `src`: the source sentence
    - `tgt`: the target sentence

```
data_pairs = []
for src, tgt in data.items():
    src_encoded = [char2ID["<BOS>"]]                           # starting from a `<BOS>` token
    for c in src:
        src_encoded.append(char2ID.get(c, char2ID["<UNK>"]))   # dictionary.get(key, default)
    src_encoded.append(char2ID["<EOS>"])                       # ending woth a `<EOS>` token

    tgt_encoded = [char2ID["<BOS>"]]                           # starting from a `<BOS>` token
    for c in tgt:
        tgt_encoded.append(char2ID.get(c, char2ID["<UNK>"]))   # dictionary.get(key, default)
    tgt_encoded.append(char2ID["<EOS>"])                       # ending woth a `<EOS>` token

    data_pairs.append((src_encoded, tgt_encoded))              # saving tuples of sentence paris in the list `data_pairs`
```

In [ ]:
data_pairs = []
for src, tgt in data.items():
    src_encoded = [char2ID["<BOS>"]]                           # starting from a `<BOS>` token
    for c in src:
        src_encoded.append(char2ID.get(c, char2ID["<UNK>"]))   # dictionary.get(key, default)
    src_encoded.append(char2ID["<EOS>"])                       # ending woth a `<EOS>` token

    tgt_encoded = [char2ID["<BOS>"]]                           # starting from a `<BOS>` token
    for c in tgt:
        tgt_encoded.append(char2ID.get(c, char2ID["<UNK>"]))   # dictionary.get(key, default)
    tgt_encoded.append(char2ID["<EOS>"])                       # ending woth a `<EOS>` token

    data_pairs.append((src_encoded, tgt_encoded))              # saving tuples of sentence paris in the list `data_pairs`

- converting data into tensors
  - for each pair in `data_pairs`:
    - `pair[0]`: the source sentence
    - `pair[1]`: the target sentence
    
```
src_tensors = []
for pair in data_pairs:
  src_tensors.append(torch.tensor(pair[0]))

tgt_tensors = []
for pair in data_pairs:
  tgt_tensors.append(torch.tensor(pair[1]))
```

In [ ]:
src_tensors = []
for pair in data_pairs:
  src_tensors.append(torch.tensor(pair[0]))

tgt_tensors = []
for pair in data_pairs:
  tgt_tensors.append(torch.tensor(pair[1]))

- padding
  - We used `pad_sequence` to automatically pad the sentence to the same length.

```
src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=char2ID["<PAD>"])
tgt_padded = pad_sequence(tgt_tensors, batch_first=True, padding_value=char2ID["<PAD>"])
```

In [ ]:
src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=char2ID["<PAD>"])
tgt_padded = pad_sequence(tgt_tensors, batch_first=True, padding_value=char2ID["<PAD>"])

- converting data into `DataLoader`

```
data_loader = DataLoader(TensorDataset(src_padded, tgt_padded), batch_size=2, shuffle=True)
```

In [ ]:
data_loader = DataLoader(TensorDataset(src_padded, tgt_padded), batch_size=2, shuffle=True)

## Encoder-Decoder




### Model defining


> Compared with the LSTM classifier last week, note that we return different items in both the encoder and the decoder this week.

- `LSTMEncoder()`:

  We output `hidden` and `cell` from the encoder, and they will be further fed to the decoder as the input.

- `LSTMDecoder()`

  We output `output` from the decoder, and they will be transferred into sentence.

  <img src="https://hackmd.io/_uploads/Bk1Nf8cAJx.jpg" width="80%">

  <small>Figure reference: [Encoder-decoder architecture for sequence generation. Adapted from Speech and Language Processing (3rd ed., Fig. 8.17, p. 177), by D. Jurafsky & J. H. Martin, 2020](https://web.stanford.edu/~jurafsky/slp3/ed3book_Jan25.pdf) </small>

```
class LSTMEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

class LSTMDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.output_transform = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_token, hidden_from_encoder, cell_from_encoder):
        input_token = input_token.unsqueeze(1)  # [B] -> [B, 1]
        embedded = self.embedding(input_token)
        output, (hidden, cell) = self.lstm(embedded, (hidden_from_encoder, cell_from_encoder))
        prediction = self.output_transform(output.squeeze(1))  # [B, output_dim]
        return prediction, hidden, cell
```


In [ ]:
class LSTMEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell
class LSTMDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.output_transform = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_token, hidden_from_encoder, cell_from_encoder):
        input_token = input_token.unsqueeze(1)  # [B] -> [B, 1]
        embedded = self.embedding(input_token)
        output, (hidden, cell) = self.lstm(embedded, (hidden_from_encoder, cell_from_encoder))
        prediction = self.output_transform(output.squeeze(1))  # [B, output_dim]
        return prediction, hidden, cell

- initialising the model

```
torch.manual_seed(4)  # the random seed of initialisation

encoder = LSTMEncoder(input_dim=len(char2ID), emb_dim=64, hidden_dim=128)
decoder = LSTMDecoder(output_dim=len(char2ID), emb_dim=64, hidden_dim=128)

# setting the loss function
criterion = nn.CrossEntropyLoss()

optimizer_lstm = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)
```

In [ ]:
torch.manual_seed(4)  # the random seed of initialisation

encoder = LSTMEncoder(input_dim=len(char2ID), emb_dim=64, hidden_dim=128)
decoder = LSTMDecoder(output_dim=len(char2ID), emb_dim=64, hidden_dim=128)

# setting the loss function
criterion = nn.CrossEntropyLoss()

optimizer_lstm = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)

### `GPU` Setting

We move both the encoder and the decoder to the `GPU`.

```
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = encoder.to(device)
decoder = decoder.to(device)

print("We are using",next(encoder.parameters()).device,".")
print("We are using",next(decoder.parameters()).device,".")
```

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = encoder.to(device)
decoder = decoder.to(device)

print("We are using",next(encoder.parameters()).device,".")
print("We are using",next(decoder.parameters()).device,".")

We are using cuda:0 .
We are using cuda:0 .


### Training

> We use teacher forcing to help the model learn more efficiently by providing the correct target token at each step during training.

```
epoch_num=50
for epoch in range(epoch_num):
    encoder.train()
    decoder.train()
    for src, tgt in data_loader:
        src = src.to(device).long()
        tgt = tgt.to(device).long()
        optimizer_lstm.zero_grad()

        hidden_from_encoder, cell_from_encoder = encoder(src)
        input_token = tgt[:, 0]           # tgt[:, 0]: for each sequence in the batch, select the token at position 0 (typically <BOS>)


        loss = 0
        for t in range(1, tgt.size(1)):
            output, hidden, cell = decoder(input_token, hidden_from_encoder, cell_from_encoder)
            loss += criterion(output, tgt[:, t])
            input_token = tgt[:, t]       # teacher forcing: we use the ground truth token as the next input for the decoder

        loss.backward()
        optimizer_lstm.step()

    # printing the loss of each epoch
    print(f"Epoch {epoch+1}/{epoch_num} - Train Loss: {loss.item():.4f}")

```


In [ ]:
epoch_num=50
for epoch in range(epoch_num):
    encoder.train()
    decoder.train()
    for src, tgt in data_loader:
        src = src.to(device).long()
        tgt = tgt.to(device).long()
        optimizer_lstm.zero_grad()

        hidden_from_encoder, cell_from_encoder = encoder(src)
        input_token = tgt[:, 0]           # tgt[:, 0]: for each sequence in the batch, select the token at position 0 (typically <BOS>)


        loss = 0
        for t in range(1, tgt.size(1)):
            output, hidden, cell = decoder(input_token, hidden_from_encoder, cell_from_encoder)
            loss += criterion(output, tgt[:, t])
            input_token = tgt[:, t]       # teacher forcing: we use the ground truth token as the next input for the decoder

        loss.backward()
        optimizer_lstm.step()

    # printing the loss of each epoch
    print(f"Epoch {epoch+1}/{epoch_num} - Train Loss: {loss.item():.4f}")

Epoch 1/50 - Train Loss: 18.3790
Epoch 2/50 - Train Loss: 13.2360
Epoch 3/50 - Train Loss: 11.8235
Epoch 4/50 - Train Loss: 5.5084
Epoch 5/50 - Train Loss: 7.4863
Epoch 6/50 - Train Loss: 4.4369
Epoch 7/50 - Train Loss: 6.0390
Epoch 8/50 - Train Loss: 4.0641
Epoch 9/50 - Train Loss: 3.8798
Epoch 10/50 - Train Loss: 5.2575
Epoch 11/50 - Train Loss: 2.1874
Epoch 12/50 - Train Loss: 2.2100
Epoch 13/50 - Train Loss: 2.4045
Epoch 14/50 - Train Loss: 1.5737
Epoch 15/50 - Train Loss: 1.5977
Epoch 16/50 - Train Loss: 1.0068
Epoch 17/50 - Train Loss: 1.5461
Epoch 18/50 - Train Loss: 2.6455
Epoch 19/50 - Train Loss: 2.3690
Epoch 20/50 - Train Loss: 0.8797
Epoch 21/50 - Train Loss: 0.8138
Epoch 22/50 - Train Loss: 1.1702
Epoch 23/50 - Train Loss: 0.4110
Epoch 24/50 - Train Loss: 0.5217
Epoch 25/50 - Train Loss: 0.2088
Epoch 26/50 - Train Loss: 0.4393
Epoch 27/50 - Train Loss: 0.2168
Epoch 28/50 - Train Loss: 0.4867
Epoch 29/50 - Train Loss: 0.2071
Epoch 30/50 - Train Loss: 0.1128
Epoch 31/50 - Tr

### Predicting and evaluating


**Note that we set the upper bound of the decoder output to avoid infinite generation if `<EOS>` is never predicted**

```
encoder.eval()
decoder.eval()


input_text = input('請輸入句子：')  # inputting the sentence

# converting sentence into IDs
beginning_token = [char2ID["<BOS>"]]

char2ID_list=[]
for c in input_text:
  char2ID_list.append(char2ID.get(c, char2ID["<UNK>"]))

ending_token = [char2ID["<EOS>"]]

input_ids= beginning_token + char2ID_list + ending_token

input_tensor = torch.tensor(input_ids).unsqueeze(0).to(device)  # [1, T]

# predicting
with torch.no_grad():
    # encoding
    hidden_from_encoder, cell_from_encoder = encoder(input_tensor)

    # decoding
    input_token = torch.tensor([char2ID["<BOS>"]]).to(device)
    outputs = []

    for _ in range(20):  # setting the upper bound
        output, hidden, cell = decoder(input_token, hidden_from_encoder, cell_from_encoder)
        pred_token = output.argmax(1)               # selecting the token with the highest probability
        if pred_token.item() == char2ID["<EOS>"]:
            break                                   # breaking the loop when the predicted token is `<EOS>`
        outputs.append(pred_token.item())
        input_token = pred_token                    # using the current output (the predicted token) to feed the decoder as the next input

# converting IDs into characters
output_text = "".join([ID2char[i] for i in outputs])
print(f"模型解釋：{output_text}")
```


In [ ]:
encoder.eval()
decoder.eval()


input_text = input('請輸入句子：')  # inputting the sentence

# converting sentence into IDs
beginning_token = [char2ID["<BOS>"]]

char2ID_list=[]
for c in input_text:
  char2ID_list.append(char2ID.get(c, char2ID["<UNK>"]))

ending_token = [char2ID["<EOS>"]]

input_ids= beginning_token + char2ID_list + ending_token

input_tensor = torch.tensor(input_ids).unsqueeze(0).to(device)  # [1, T]

# predicting
with torch.no_grad():
    # encoding
    hidden_from_encoder, cell_from_encoder = encoder(input_tensor)

    # decoding
    input_token = torch.tensor([char2ID["<BOS>"]]).to(device)
    outputs = []

    for _ in range(20):  # setting the upper bound
        output, hidden, cell = decoder(input_token, hidden_from_encoder, cell_from_encoder)
        pred_token = output.argmax(1)               # selecting the token with the highest probability
        if pred_token.item() == char2ID["<EOS>"]:
            break                                   # breaking the loop when the predicted token is `<EOS>`
        outputs.append(pred_token.item())
        input_token = pred_token                    # using the current output (the predicted token) to feed the decoder as the next input

# converting IDs into characters
output_text = "".join([ID2char[i] for i in outputs])
print(f"模型解釋：{output_text}")

請輸入句子：我覺得不行
模型解釋：我覺得其實討厭


## The transformer



### Model defining
Unlike an RNN model, a Transformer does not inherently consider the order of tokens within an input sentence. Therefore, we need to explicitly inject positional information to `positional_embeddings`.

Afterwards, we feed them to the Transformer to work with token embeddings.

- obtaining positional embeddings
- building up a Transformer

#### Obtaining positional embeddings

In practice, we first generate the positional embeddings and then combine them with token embeddings, which are trained within the Transformer.  
To compute the positional embeddings, we use:

- `position`: assigning a unique index to each token based on its position in the sequence  
- `div_term`: controlling the frequency of the sine and cosine waves used to encode each position, based on `d_model`


<img src="https://hackmd.io/_uploads/SJE06ssAkg.jpg" width="80%">

  <small>Figure reference: [Sinusoidal Patterns Across Dimensions. YouTube: How positional encoding works in transformers?. BrainDrain, uploaded on 17.12.2023.](https://youtu.be/T3OT8kqoqjc?si=ddriH_uYBJ6wQ4rM) </small>

```
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        """
        d_model: determining the dimension number of the positional embeddings
        max_len: determining the maximum length of the input sequence
        """
        super().__init__()

        pe = torch.zeros(max_len, d_model)           # creating a tensor to save positional embeddings

        position = torch.arange(0, max_len).unsqueeze(1).float() # creating the list of position indices
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term) # starting from position 0, going to the end, and taking every 2nd element
        pe[:, 1::2] = torch.cos(position * div_term) # starting from position 1, going to the end, and taking every 2nd element
        self.register_buffer('pe', pe.unsqueeze(1))

    def forward(self, token_embeddings):
        return token_embeddings + self.pe[:token_embeddings.size(0)]
```


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        """
        d_model: determining the dimension number of the positional embeddings
        max_len: determining the maximum length of the input sequence
        """
        super().__init__()

        pe = torch.zeros(max_len, d_model)           # creating a tensor to save positional embeddings

        position = torch.arange(0, max_len).unsqueeze(1).float() # creating the list of position indices
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term) # starting from position 0, going to the end, and taking every 2nd element
        pe[:, 1::2] = torch.cos(position * div_term) # starting from position 1, going to the end, and taking every 2nd element
        self.register_buffer('pe', pe.unsqueeze(1))

    def forward(self, token_embeddings):
        return token_embeddings + self.pe[:token_embeddings.size(0)]

#### Building up the Transformer

In the Transformer, we first perform an element-wise addition of positional and token embeddings. Afterwards, we utilise the combined embeddings as input to the encoder and decoder layers for further processing.

```
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len):
        """
        vocab_size: determining the size of the vocabulary
        d_model: determining the dimension number of the positional embeddings
        nhead: determining the number of heads in the multihead attention
        num_layers: determining the number of layers in the encoder and decoder
        max_len: determining the maximum length of the input sequence
        """
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=char2ID["<PAD>"])
        self.pos_encoder = PositionalEncoding(d_model,max_len)
        self.transformer = nn.Transformer(d_model=d_model,                  
                                          nhead=nhead,                      
                                          num_encoder_layers=num_layers,    
                                          num_decoder_layers=num_layers,    
                                          batch_first=True)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src_ID, tgt_ID):
        src_token_embeddings = self.embedding(src_ID)  
        tgt_token_embeddings = self.embedding(tgt_ID)
        src_all_embeddings = self.pos_encoder(src_token_embeddings)
        tgt_all_embeddings = self.pos_encoder(tgt_token_embeddings)
        output = self.transformer(src_all_embeddings, tgt_all_embeddings)
        return self.out(output)
```

In [ ]:
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len):
        """
        vocab_size: determining the size of the vocabulary
        d_model: determining the dimension number of the positional embeddings
        nhead: determining the number of heads in the multihead attention
        num_layers: determining the number of layers in the encoder and decoder
        max_len: determining the maximum length of the input sequence
        """
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=char2ID["<PAD>"])
        self.pos_encoder = PositionalEncoding(d_model,max_len)
        self.transformer = nn.Transformer(d_model=d_model,
                                          nhead=nhead,
                                          num_encoder_layers=num_layers,
                                          num_decoder_layers=num_layers,
                                          batch_first=True)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src_ID, tgt_ID):
        src_token_embeddings = self.embedding(src_ID)
        tgt_token_embeddings = self.embedding(tgt_ID)
        src_all_embeddings = self.pos_encoder(src_token_embeddings)
        tgt_all_embeddings = self.pos_encoder(tgt_token_embeddings)
        output = self.transformer(src_all_embeddings, tgt_all_embeddings)
        return self.out(output)

- calculating the maximum length of the input sentences

To ensure that the positional embedding table is sufficiently large, we first calculate the maximum sequence length across both the source and target sentences. This step is essential when using Transformer models, as they require positional encodings to provide tokens with information about their positions within a sequence.

```
# max_src_len = max(len(seq) for seq in data.keys())
src_len=[]
for seq in data.keys():
  src_len.append(len(seq))
max_src_len=max(src_len)


# max_tgt_len = max(len(seq) for seq in data.values())
tgt_len=[]
for seq in data.values():
  tgt_len.append(len(seq))
max_tgt_len=max(tgt_len)

max_length = max(max_src_len, max_tgt_len) + 2  # +2 for BOS/EOS if needed
print(max_length)
```

In [ ]:
# max_src_len = max(len(seq) for seq in data.keys())
src_len=[]
for seq in data.keys():
  src_len.append(len(seq))
max_src_len=max(src_len)


# max_tgt_len = max(len(seq) for seq in data.values())
tgt_len=[]
for seq in data.values():
  tgt_len.append(len(seq))
max_tgt_len=max(tgt_len)

max_length = max(max_src_len, max_tgt_len) + 2  # +2 for BOS/EOS if needed
print(max_length)

10


- initialising the model

```
transformer_model = SimpleTransformer(
    vocab_size=len(char2ID),
    d_model=16,
    nhead=8,
    num_layers=1,
    max_len=max_length
)
optimizer_transformer = optim.Adam(transformer_model.parameters(), lr=0.001)
```

In [ ]:
transformer_model = SimpleTransformer(
    vocab_size=len(char2ID),
    d_model=16,
    nhead=8,
    num_layers=1,
    max_len=max_length
)
optimizer_transformer = optim.Adam(transformer_model.parameters(), lr=0.001)

### `GPU` Setting

It is common to run deep learning models on a `GPU` to save both time and memory.  
- It is necessary to specify the `GPU` device on which the model should run; otherwise, the model will run on the `CPU` by default.

```
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer_model = transformer_model.to(device)
print("We are using",next(transformer_model.parameters()).device,".")
```

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer_model = transformer_model.to(device)
print("We are using",next(transformer_model.parameters()).device,".")

We are using cuda:0 .


### Training

Although the term teacher forcing is more commonly associated with LSTM-based encoder-decoder, the same concept is applied in Transformers through the use of shifted target sequences as decoder inputs. This allows the model to learn from the ground truth context at each training step.

- `tgt_input`: excluding the last token and serving as input to the decoder

  > `[<BOS>, '我', '要', '工', '作']`
- `tgt_target`: excluding the first token and representing the expected output
  
  > `['我', '要', '工', '作', <EOS>]`

```
epoch_num=50
for epoch in range(epoch_num):
    transformer_model.train()
    for src, tgt in data_loader:
        src = src.to(device).long()  
        tgt = tgt.to(device).long()  
        tgt_input = tgt[:, :-1]
        tgt_target = tgt[:, 1:]

        optimizer_transformer.zero_grad()
        output = transformer_model(src, tgt_input)
        output = output.view(-1, output.shape[-1])
        tgt_target = tgt_target.reshape(-1)
        loss = criterion(output, tgt_target)
        loss.backward()
        optimizer_transformer.step()

    print(f"Epoch {epoch+1}/{epoch_num} - Train Loss: {loss.item():.4f}")
```


In [ ]:
epoch_num=50
for epoch in range(epoch_num):
    transformer_model.train()
    for src, tgt in data_loader:
        src = src.to(device).long()
        tgt = tgt.to(device).long()
        tgt_input = tgt[:, :-1]
        tgt_target = tgt[:, 1:]

        optimizer_transformer.zero_grad()
        output = transformer_model(src, tgt_input)
        output = output.view(-1, output.shape[-1])
        tgt_target = tgt_target.reshape(-1)
        loss = criterion(output, tgt_target)
        loss.backward()
        optimizer_transformer.step()

    print(f"Epoch {epoch+1}/{epoch_num} - Train Loss: {loss.item():.4f}")

Epoch 1/50 - Train Loss: 3.0093
Epoch 2/50 - Train Loss: 2.5097
Epoch 3/50 - Train Loss: 1.8805
Epoch 4/50 - Train Loss: 1.8203
Epoch 5/50 - Train Loss: 1.4362
Epoch 6/50 - Train Loss: 1.2705
Epoch 7/50 - Train Loss: 1.5609
Epoch 8/50 - Train Loss: 0.6984
Epoch 9/50 - Train Loss: 0.6739
Epoch 10/50 - Train Loss: 0.7823
Epoch 11/50 - Train Loss: 0.7834
Epoch 12/50 - Train Loss: 0.6207
Epoch 13/50 - Train Loss: 0.5453
Epoch 14/50 - Train Loss: 0.5452
Epoch 15/50 - Train Loss: 0.8094
Epoch 16/50 - Train Loss: 0.5417
Epoch 17/50 - Train Loss: 0.4345
Epoch 18/50 - Train Loss: 0.2406
Epoch 19/50 - Train Loss: 0.0887
Epoch 20/50 - Train Loss: 0.6228
Epoch 21/50 - Train Loss: 0.4089
Epoch 22/50 - Train Loss: 0.2944
Epoch 23/50 - Train Loss: 0.3031
Epoch 24/50 - Train Loss: 0.2600
Epoch 25/50 - Train Loss: 0.3653
Epoch 26/50 - Train Loss: 0.2545
Epoch 27/50 - Train Loss: 0.2383
Epoch 28/50 - Train Loss: 0.1126
Epoch 29/50 - Train Loss: 0.1670
Epoch 30/50 - Train Loss: 0.0969
Epoch 31/50 - Train

### Predicting and evaluating


```
transformer_model.eval()

input_text = input("請輸入句子（Transformer）：")

beginning_token = [char2ID["<BOS>"]]
char2ID_list = [char2ID.get(c, char2ID["<UNK>"]) for c in input_text]
ending_token = [char2ID["<EOS>"]]
input_ids = beginning_token + char2ID_list + ending_token

src_tensor = torch.tensor(input_ids).unsqueeze(0).to(device)

with torch.no_grad():
    generated = [char2ID["<BOS>"]]
    for _ in range(20):
        tgt_tensor = torch.tensor(generated).unsqueeze(0).to(device)
        output = transformer_model(src_tensor, tgt_tensor)
        next_token = output[0, -1].argmax(dim=-1).item()
        if next_token == char2ID["<EOS>"]:
            break
        generated.append(next_token)

decoded = "".join([ID2char[i] for i in generated[1:]])
print(f"Transformer 解釋：{decoded}")
```


In [ ]:
transformer_model.eval()

input_text = input("請輸入句子（Transformer）：")

beginning_token = [char2ID["<BOS>"]]
char2ID_list = [char2ID.get(c, char2ID["<UNK>"]) for c in input_text]
ending_token = [char2ID["<EOS>"]]
input_ids = beginning_token + char2ID_list + ending_token

src_tensor = torch.tensor(input_ids).unsqueeze(0).to(device)

with torch.no_grad():
    generated = [char2ID["<BOS>"]]
    for _ in range(20):
        tgt_tensor = torch.tensor(generated).unsqueeze(0).to(device)
        output = transformer_model(src_tensor, tgt_tensor)
        next_token = output[0, -1].argmax(dim=-1).item()
        if next_token == char2ID["<EOS>"]:
            break
        generated.append(next_token)

decoded = "".join([ID2char[i] for i in generated[1:]])
print(f"Transformer 解釋：{decoded}")

請輸入句子（Transformer）：我覺得不行
Transformer 解釋：我覺得真的希望


# Multidimensional Slicing

Multidimensional slicing is a way to process specific elements across multiple dimensions of a data structure, such as a list of lists, a matrix, or a tensor. In deep learning, it's commonly used to extract specific values from batches of sequences. It is noted that this type of slicing `[:, 0]` is only supported by tensors or NumPy arrays—not regular Python lists.

For example, if we have a tensor representing 3 sequences, each with 4 tokens:

```
# 3D tensor
x = torch.tensor([
    [[10, 11], [12, 13]], # x[0]: 1st tensor of the 1st dimension
    [[20, 21], [22, 23]], # x[1]: 2nd tensor of the 1st dimension
    [[30, 31], [32, 33]]  # x[2]: 3rd tensor of the 1st dimension
])
```




In [ ]:
# 3D tensor
x = torch.tensor([
    [[10, 11], [12, 13]], # x[0]: 1st tensor of the 1st dimension
    [[20, 21], [22, 23]], # x[1]: 2nd tensor of the 1st dimension
    [[30, 31], [32, 33]]  # x[2]: 3rd tensor of the 1st dimension
])
x

tensor([[[10, 11],
         [12, 13]],

        [[20, 21],
         [22, 23]],

        [[30, 31],
         [32, 33]]])

- index of the tensors

```
# x[<1st dimension>,<2nd dimension>,<3rd dimension>]
x[0]
x[0, 0]
x[0, 0, 0]
x[:]
x[:,0]
x[:,:,0]
```

In [ ]:
# x[<1st dimension>,<2nd dimension>,<3rd dimension>]
# x[0]
# x[0, 0]
# x[0, 0, 0]
# x[:]
x[:,0]
# x[:,:,0]

tensor([[10, 11],
        [20, 21],
        [30, 31]])

# Assignment

Please create a new `.ipynb` file to complete your assignment. Don't forget to include your name and relevant information at the top of your code.


1. `BiLSTM-based Encoder-Decoder`

  - Build a `BiLSTM-based Encoder-Decoder` and go through training and predicting. (30%)
  - Do you think `BiLSTM-based Encoder-Decoder` work better than the `LSTM-based Encoder-Decoder` in class? Why or why not? (20%)

2. `Transformer`

  - Adjust more than two hyperparameters of the `Transformer` and explain the reasons for your adjustment. (20%)  

  - After making your adjustments, do you think the model you train work better? Why or why not? (20%)


**Bonus1:** Try to conduct a seq2seq task on your own dataset. Explain the reason you work on this dataset and why it is worth trying.
